![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

In [1]:
# Clear all previous variables from memory
import gc

# Get list of all variables before loading
_vars_to_keep = ['gc', 'pd', 'np']  # Keep essential imports

# Delete all user-defined variables
_all_vars = list(globals().keys())
for _var in _all_vars:
    if not _var.startswith('_') and _var not in _vars_to_keep:
        try:
            del globals()[_var]
        except:
            pass

# Aggressive garbage collection
gc.collect()
gc.collect()
gc.collect()

print("✓ Memory cleared")

# Re-import if needed
import pandas as pd
import numpy as np

# Load both parquet files efficiently
print("\nLoading data...")

cryptos_price = pd.read_parquet('output.parquet')
# Remove BTC OHLCV if present
btc_ohlcv_features = ['BTCUSDT_open', 'BTCUSDT_high', 'BTCUSDT_low', 'BTCUSDT_close', 'BTCUSDT_volume']
cryptos_price = cryptos_price.drop(columns=[col for col in btc_ohlcv_features if col in cryptos_price.columns], errors='ignore')

btc_futures = pd.read_parquet('btc_futures_clean.parquet', engine='pyarrow') 
commodities = pd.read_parquet('commodities_clean.parquet', engine='pyarrow')
forex = pd.read_parquet('forex_merged.parquet', engine='pyarrow')
indices = pd.read_parquet('indices_all_merged.parquet', engine='pyarrow')

print(f"✓ Cryptos: {cryptos_price.shape}, {cryptos_price.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"✓ Futures: {btc_futures.shape}, {btc_futures.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"✓ Commodities: {commodities.shape}, {commodities.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"✓ Forex: {forex.shape}, {forex.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"✓ Indices: {indices.shape}, {indices.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Clean up temporary variables
del _vars_to_keep, _all_vars, _var
gc.collect()

In [ ]:
print("First index:")
print(f"cryptos_price: {cryptos_price.index[0]}")
print(f"btc_futures:   {btc_futures.index[0]}")
print(f"commodities:   {commodities.index[0]}")
print(f"forex:         {forex.index[0]}")
print(f"indices:       {indices.index[0]}")

In [ ]:
print("="*80)
print("NaN ANALYSIS FOR EACH DATASET")
print("="*80)

datasets = {
    'Cryptos': cryptos_price,
    "Futures": btc_futures,
    'Commodities': commodities,
    'Forex': forex,
    'Indices': indices
}

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(f"  Shape: {df.shape}")
    print(f"  Total cells: {df.shape[0] * df.shape[1]:,}")
    print(f"  Total NaNs: {df.isna().sum().sum():,}")
    print(f"  NaN percentage: {df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100:.2f}%")
    print(f"  Rows with ANY NaN: {df.isna().any(axis=1).sum():,} ({df.isna().any(axis=1).sum()/len(df)*100:.2f}%)")
    print(f"  Columns with NaNs: {(df.isna().sum() > 0).sum()} / {df.shape[1]}")
    
    # Top 5 columns with most NaNs
    nan_cols = df.isna().sum().sort_values(ascending=False).head(5)
    if nan_cols.sum() > 0:
        print(f"  Top 5 columns with NaNs:")
        for col, count in nan_cols.items():
            if count > 0:
                print(f"    - {col}: {count:,} ({count/len(df)*100:.2f}%)")

Target

In [ ]:
# Rename BTCUSDT_ prefixed metric columns
rename_map = {
    'BTCUSDT_total_return': 'total_return',
    'BTCUSDT_sharpe': 'sharpe',
    'BTCUSDT_sortino': 'sortino',
    'BTCUSDT_upr': 'upr'
}

cryptos_price = cryptos_price.rename(columns=rename_map)
print(f"✓ Renamed metric columns to remove BTCUSDT_ prefix")

print("\n" + "="*80)
print("CREATING TARGET COLUMNS")
print("="*80)

# ===== CREATE TARGET COLUMNS WITH THRESHOLDS =====
cryptos_price['target_n3'] = (cryptos_price['total_return'] > 0.0).astype(int)
cryptos_price['target_n3_001'] = (cryptos_price['total_return'] > 0.001).astype(int)
cryptos_price['target_sharpe_n3_10'] = (cryptos_price['sharpe'] > 1.0).astype(int)
cryptos_price['target_sharpe_n3_20'] = (cryptos_price['sharpe'] > 2.0).astype(int)
cryptos_price['target_sortino_n3_05'] = (cryptos_price['sortino'] > 0.5).astype(int)
cryptos_price['target_sortino_n3_15'] = (cryptos_price['sortino'] > 1.5).astype(int)
cryptos_price['target_upr_n3_13'] = (cryptos_price['upr'] > 1.3).astype(int)

print("✓ Created 7 target columns")

# ===== PRINT CLASS BALANCE =====
print("\n" + "="*80)
print("TARGET CLASS BALANCE")
print("="*80)

target_cols = [
    'target_n3',
    'target_n3_001',
    'target_sharpe_n3_10',
    'target_sharpe_n3_20',
    'target_sortino_n3_05',
    'target_sortino_n3_15',
    'target_upr_n3_13'
]

for target in target_cols:
    valid_mask = cryptos_price[target].notna()
    if valid_mask.sum() > 0:
        pos_pct = cryptos_price.loc[valid_mask, target].mean() * 100
        n_pos = int(cryptos_price.loc[valid_mask, target].sum())
        n_total = valid_mask.sum()
        print(f"{target:25s}: {pos_pct:5.2f}% positive ({n_pos:,} / {n_total:,} samples)")
    else:
        print(f"{target:25s}: No valid samples")

In [ ]:
print("\n" + "="*80)
print("REMOVING RAW METRIC COLUMNS")
print("="*80)

# Remove raw metrics to prevent data leakage
raw_metrics = ['total_return', 'sharpe', 'sortino', 'upr']
cryptos_price = cryptos_price.drop(columns=raw_metrics, errors='ignore')

print(f"✓ Removed: {', '.join(raw_metrics)}")
print(f"✓ Final shape: {cryptos_price.shape}")
print(f"✓ Targets preserved: {[col for col in cryptos_price.columns if col.startswith('target_')]}")

Create common_indices and full dataset.

In [ ]:
# Group datasets
datasets = [commodities, forex, indices, btc_futures]

# Dataset 1: Only common indices
common_index = cryptos_price.index
for df in datasets:
    common_index = common_index.intersection(df.index)

merged_common = pd.concat([
    cryptos_price.loc[common_index],
    btc_futures.loc[common_index],  # Changed: Added .loc here
    commodities.loc[common_index],
    forex.loc[common_index],
    indices.loc[common_index]
], axis=1)

print("Dataset 1 - Common indices only:")
print(f"  Timestamps: {len(common_index)}")
print(f"  Date range: {common_index.min()} to {common_index.max()}")
print(f"  Shape: {merged_common.shape}")

"""
# Dataset 2: All crypto_price dates (NO FORWARD FILL - keep NaNs for XGBoost)
merged_all = cryptos_price.copy()
for df in datasets:
    merged_all = merged_all.join(df, how='left')

print("\nDataset 2 - All crypto dates (with NaNs, no forward fill):")
print(f"  Timestamps: {len(merged_all)}")
print(f"  Date range: {merged_all.index.min()} to {merged_all.index.max()}")
print(f"  Shape: {merged_all.shape}")
print(f"  NaNs: {merged_all.isna().sum().sum():,}")

del cryptos_price, commodities, forex, indices, datasets, df, btc_futures
gc.collect()
"""

In [ ]:
import gc
import pandas as pd

# ===== MEMORY-OPTIMIZED DATASET 2: CHUNKED JOINING =====

print("="*80)
print("DATASET 2: CHUNKED JOINING (MEMORY OPTIMIZED)")
print("="*80)

# Step 1: Optimize dtypes before joining (float64 -> float32 = 50% memory saving)
def optimize_dtypes(df):
    """Convert float64 to float32 to save memory"""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype('float32')
    return df

print("Step 1: Optimizing dtypes...")
cryptos_price = optimize_dtypes(cryptos_price)
commodities = optimize_dtypes(commodities)
forex = optimize_dtypes(forex)
indices = optimize_dtypes(indices)
btc_futures = optimize_dtypes(btc_futures)

# Step 2: Join sequentially and free memory after each step
print("\nStep 2: Sequential joining with memory cleanup...")

# Start with crypto prices
merged_all = cryptos_price.copy()
del cryptos_price
gc.collect()
print(f"  After crypto:       {merged_all.memory_usage(deep=True).sum() / 1024**2:>8.1f} MB | Shape: {merged_all.shape}")

# Join BTC futures
merged_all = merged_all.join(btc_futures, how='left')
del btc_futures
gc.collect()
print(f"  After BTC futures:  {merged_all.memory_usage(deep=True).sum() / 1024**2:>8.1f} MB | Shape: {merged_all.shape}")

# Join commodities
merged_all = merged_all.join(commodities, how='left')
del commodities
gc.collect()
print(f"  After commodities:  {merged_all.memory_usage(deep=True).sum() / 1024**2:>8.1f} MB | Shape: {merged_all.shape}")

# Join forex
merged_all = merged_all.join(forex, how='left')
del forex
gc.collect()
print(f"  After forex:        {merged_all.memory_usage(deep=True).sum() / 1024**2:>8.1f} MB | Shape: {merged_all.shape}")

# Join indices
merged_all = merged_all.join(indices, how='left')
del indices
gc.collect()
print(f"  After indices:      {merged_all.memory_usage(deep=True).sum() / 1024**2:>8.1f} MB | Shape: {merged_all.shape}")

print("\nDataset 2 - All crypto dates (with NaNs, no forward fill):")
print(f"  Timestamps: {len(merged_all):,}")
print(f"  Date range: {merged_all.index.min()} to {merged_all.index.max()}")
print(f"  Shape: {merged_all.shape}")
print(f"  NaNs: {merged_all.isna().sum().sum():,}")
print(f"  Final memory: {merged_all.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Final cleanup
gc.collect()

### FEATURE SELECTION: Variance Threshold, Correlatio nfilter and Mutual INformation


REMOVE CORRELATED FEATURES

In [ ]:
import numpy as np
import pandas as pd
import gc

# ===== CORRELATION FILTER ON FULL DATASET =====
print("="*80)
print("CORRELATION ANALYSIS & FILTERING")
print("="*80)

dataset = merged_all.copy()

# Get numeric columns (exclude ALL targets)
target_cols = [
    'target_n3',
    'target_n3_001',
    'target_sharpe_n3_10',
    'target_sharpe_n3_20',
    'target_sortino_n3_05',
    'target_sortino_n3_15',
    'target_upr_n3_13'
]
numeric_cols = [col for col in dataset.select_dtypes(include=[np.number]).columns 
                if col not in target_cols]

print(f"Analyzing {len(numeric_cols)} features (excluding {len(target_cols)} targets)")

# Sample data for correlation computation
if len(dataset) > 15000:
    valid_rows = dataset['BTCUSDT_ret_3'].notna()  # Changed to ret_3
    sample_indices = dataset[valid_rows].sample(n=min(15000, valid_rows.sum()), random_state=42).index
    sample_data = dataset.loc[sample_indices, numeric_cols]
else:
    sample_data = dataset[numeric_cols]

print(f"Processing {len(numeric_cols)} columns on {len(sample_data)} rows")

# Compute correlations
print("Computing correlation matrix...")
corr_matrix = sample_data.corr()

print("Computing correlation with BTCUSDT_ret_3...")
btc_returns_corr = sample_data.corrwith(sample_data['BTCUSDT_ret_3']).abs()  # Changed to ret_3

# Find high correlation pairs
threshold = 0.85
high_corr_pairs = []

print(f"Finding pairs with |correlation| > {threshold}...")
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        corr_val = corr_matrix.iloc[i, j]
        if abs(corr_val) > threshold:
            high_corr_pairs.append((numeric_cols[i], numeric_cols[j], corr_val))

print(f"Found {len(high_corr_pairs)} pairs with |correlation| > {threshold}")

# Decide which features to drop
protected_cols = ['BTCUSDT_ret_3']  # Protect the correlation reference
features_to_drop = set()

for col1, col2, corr_val in high_corr_pairs:
    corr1 = btc_returns_corr.get(col1, 0)
    corr2 = btc_returns_corr.get(col2, 0)
    
    # Drop the one with lower correlation to BTCUSDT_ret_3
    candidate_to_drop = col2 if corr1 >= corr2 else col1
    
    if candidate_to_drop not in protected_cols:
        features_to_drop.add(candidate_to_drop)
    elif col1 not in protected_cols:
        features_to_drop.add(col1)
    elif col2 not in protected_cols:
        features_to_drop.add(col2)

features_to_drop = list(features_to_drop)

# Apply to both datasets
print(f"\nRemoving {len(features_to_drop)} redundant features from both datasets...")
print(f"  merged_all:    {merged_all.shape} → ", end="")
merged_all_clean = merged_all.drop(columns=features_to_drop, errors='ignore')
print(f"{merged_all_clean.shape}")

print(f"  merged_common: {merged_common.shape} → ", end="")
merged_common_clean = merged_common.drop(columns=features_to_drop, errors='ignore')
print(f"{merged_common_clean.shape}")

# Quick verification
common_cols = set(merged_common_clean.columns)
full_cols = set(merged_all_clean.columns)
print(f"\n✓ Column overlap: {len(common_cols & full_cols)}/{len(common_cols)} in common dataset")

# Verify targets are preserved
targets_in_full = [col for col in merged_all_clean.columns if 'target' in col]
targets_in_common = [col for col in merged_common_clean.columns if 'target' in col]
print(f"✓ Targets preserved in merged_all_clean: {len(targets_in_full)}")
print(f"✓ Targets preserved in merged_common_clean: {len(targets_in_common)}")

# Cleanup
del dataset, numeric_cols, sample_data, corr_matrix, btc_returns_corr
del high_corr_pairs, protected_cols, features_to_drop, common_cols, full_cols
gc.collect()

print("\n✓ Cleaned datasets ready for MI selection")

In [ ]:
import gc
import pandas as pd
import numpy as np

# ===== MEMORY CLEANUP BEFORE MI SELECTION =====
print("="*80)
print("MEMORY CLEANUP BEFORE MI SELECTION")
print("="*80)

# List of variables you MUST keep for MI selection
KEEP_VARS = {
    'merged_common_clean',  # Dataset 1: market hours
    'merged_all_clean',     # Dataset 2: 24/7
    'target_cols',          # List of all target column names
    'selected_targets',     # List of targets to compute MI for
}

# Get all current variables
all_vars = set(dir())

# Variables to delete (everything except KEEP_VARS and Python builtins)
python_builtins = set(dir(__builtins__))
to_delete = all_vars - KEEP_VARS - python_builtins - {'to_delete', 'all_vars', 'python_builtins', 'KEEP_VARS'}

# Remove common safe names that shouldn't be deleted
safe_names = {'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', 
              'np', 'pd', 'gc', 'plt', 'sns', 'warnings', 'sys', 'os'}
to_delete = to_delete - safe_names

print(f"Keeping {len(KEEP_VARS)} essential variables:")
for var in sorted(KEEP_VARS):
    if var in globals():
        try:
            size_mb = sys.getsizeof(globals()[var]) / 1024**2
            print(f"  ✓ {var}: {size_mb:.1f} MB")
        except:
            print(f"  ✓ {var}")

print(f"\nDeleting {len(to_delete)} intermediate variables...")
deleted_count = 0
for var_name in to_delete:
    if var_name in globals():
        del globals()[var_name]
        deleted_count += 1

print(f"  Deleted {deleted_count} variables")

# Force garbage collection
gc.collect()

print("\n✓ Memory cleanup complete")
print(f"✓ Ready for MI selection with {len(KEEP_VARS)} essential variables")

# Verify essential variables exist
print("\nVerifying essential variables:")
for var in KEEP_VARS:
    if var in globals():
        print(f"  ✓ {var} exists")
    else:
        print(f"  ❌ {var} MISSING!")

MUTUAL INFORMATION: to do next divide between regular and market hurs feautes


In [ ]:
from sklearn.feature_selection import mutual_info_classif
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

# ===== MUTUAL INFORMATION FEATURE SELECTION =====
print("="*80)
print("MUTUAL INFORMATION FEATURE SELECTION (PER TARGET)")
print("="*80)

# All targets in the dataset
target_cols = [
    'target_n3',
    'target_n3_001',
    'target_sharpe_n3_10',
    'target_sharpe_n3_20',
    'target_sortino_n3_05',
    'target_sortino_n3_15',
    'target_upr_n3_13', 
]

# Selected targets to compute MI for
selected_targets = [
    'target_n3',
    'target_n3_001',
    #'target_sharpe_n3_10',
    #'target_sharpe_n3_20',
    #'target_sortino_n3_05',
    'target_sortino_n3_15',
    'target_upr_n3_13',
]

print(f"Total targets: {len(target_cols)}")
print(f"Computing MI for: {len(selected_targets)} targets")
print(f"  {selected_targets}\n")

n_runs = 10

# Store results for all targets
all_target_results = {}

# Loop through ONLY selected targets
for target_name in selected_targets:
    print("\n" + "="*80)
    print(f"TARGET: {target_name}")
    print("="*80)
    
    # ===== DATASET 1: MARKET HOURS =====
    dataset_market = merged_common_clean.copy()
    y_market = dataset_market[target_name]
    X_market = dataset_market.drop(columns=target_cols, errors='ignore').select_dtypes(include=[np.number])
    X_market = X_market.replace([np.inf, -np.inf], np.nan)
    
    # Prepare valid data
    X_market_valid = X_market[y_market.notna()]
    y_market_valid = y_market[y_market.notna()]
    
    print(f"\nMARKET HOURS - Rows: {len(X_market_valid):,}, Class balance: {y_market_valid.mean()*100:.1f}% positive")
    
    # Adjust sample size
    sample_size_market = min(5000, int(len(X_market_valid) * 0.7))
    
    # Run MI and store BOTH counts and scores
    market_mi_scores = {}  # {feature: [list of MI scores]}
    market_appearances = {}  # {feature: count of MI > 0}
    
    for i in range(n_runs):
        sample_idx = np.random.RandomState(42+i).choice(len(X_market_valid), sample_size_market, replace=False)
        X_sample = X_market_valid.iloc[sample_idx].dropna()
        y_sample = y_market_valid.iloc[sample_idx].loc[X_sample.index]
        
        mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42+i)
        
        for feature, score in zip(X_sample.columns, mi_scores):
            # Store actual MI score
            if feature not in market_mi_scores:
                market_mi_scores[feature] = []
            market_mi_scores[feature].append(score)
            
            # Count appearances (MI > 0)
            market_appearances[feature] = market_appearances.get(feature, 0) + (1 if score > 0 else 0)
    
    
    # ===== DATASET 2: 24/7 =====
    dataset_247 = merged_all_clean.copy()
    y_247 = dataset_247[target_name]
    X_247 = dataset_247.drop(columns=target_cols, errors='ignore').select_dtypes(include=[np.number])
    X_247 = X_247.replace([np.inf, -np.inf], np.nan)
    
    # Filter to features with <10% NaN
    nan_threshold = 10
    nan_pct_247 = (X_247.isna().sum() / len(X_247) * 100)
    high_coverage_cols = nan_pct_247[nan_pct_247 <= nan_threshold].index.tolist()
    
    X_247_filtered = X_247[high_coverage_cols]
    
    # Prepare valid data
    X_247_valid = X_247_filtered[y_247.notna()]
    y_247_valid = y_247[y_247.notna()]
    
    print(f"24/7 - Rows: {len(X_247_valid):,}, Class balance: {y_247_valid.mean()*100:.1f}% positive")
    
    # Sample size
    sample_size_247 = min(10000, int(len(X_247_valid) * 0.5))
    
    # Run MI and store BOTH counts and scores
    full_mi_scores = {}  # {feature: [list of MI scores]}
    full_appearances = {}  # {feature: count of MI > 0}
    
    for i in range(n_runs):
        sample_idx = np.random.RandomState(42+i).choice(len(X_247_valid), sample_size_247, replace=False)
        X_sample = X_247_valid.iloc[sample_idx].dropna()
        y_sample = y_247_valid.iloc[sample_idx].loc[X_sample.index]
        
        mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42+i)
        
        for feature, score in zip(X_sample.columns, mi_scores):
            # Store actual MI score
            if feature not in full_mi_scores:
                full_mi_scores[feature] = []
            full_mi_scores[feature].append(score)
            
            # Count appearances (MI > 0)
            full_appearances[feature] = full_appearances.get(feature, 0) + (1 if score > 0 else 0)
    
    
    # ===== COMBINE RESULTS FOR THIS TARGET =====
    market_only_features = set(market_appearances.keys()) - set(full_appearances.keys())
    crypto_247_features = set(market_appearances.keys()) & set(full_appearances.keys())
    
    print(f"\nFeature distribution:")
    print(f"  Market-hours only: {len(market_only_features)}")
    print(f"  Crypto (24/7 validated): {len(crypto_247_features)}")
    
    # Create stability dataframe for this target
    all_features = []
    for feature in set(market_appearances.keys()) | set(full_appearances.keys()):
        market_count = market_appearances.get(feature, 0)
        full_count = full_appearances.get(feature, 0)
        
        # Get MI scores
        market_mi_list = market_mi_scores.get(feature, [])
        full_mi_list = full_mi_scores.get(feature, [])
        
        # Calculate average MI scores
        avg_mi_market = np.mean(market_mi_list) if len(market_mi_list) > 0 else 0
        avg_mi_247 = np.mean(full_mi_list) if len(full_mi_list) > 0 else 0
        
        # Determine dataset label and stability
        if feature in crypto_247_features:
            dataset_label = 'crypto_247'
            stability_pct = (market_count + full_count) / (2 * n_runs) * 100
            avg_mi_combined = (avg_mi_market + avg_mi_247) / 2
        elif feature in market_only_features:
            dataset_label = 'market_only'
            stability_pct = market_count / n_runs * 100
            avg_mi_combined = avg_mi_market
        else:
            dataset_label = '247_only'
            stability_pct = full_count / n_runs * 100
            avg_mi_combined = avg_mi_247
        
        all_features.append({
            'feature': feature,
            'market_appearances': market_count,
            'full_appearances': full_count,
            'stability_pct': stability_pct,
            'avg_mi_market': avg_mi_market,
            'avg_mi_247': avg_mi_247,
            'avg_mi_combined': avg_mi_combined,
            'dataset': dataset_label
        })
    
    stability_df = pd.DataFrame(all_features)
    
    # Sort by combined score: stability AND MI
    # Higher is better for both
    stability_df['combined_score'] = stability_df['stability_pct'] * stability_df['avg_mi_combined']
    stability_df = stability_df.sort_values('combined_score', ascending=False)
    
    # Top 10 MARKET ONLY
    print("\nTop 10 MARKET-HOURS ONLY features:")
    market_only_top = stability_df[stability_df['dataset']=='market_only'].head(25)
    for idx, row in market_only_top.iterrows():
        print(f"  {row['feature']:45s} stability: {row['stability_pct']:5.0f}% ({row['market_appearances']}/{n_runs}), avg_MI: {row['avg_mi_market']:.5f}")
    
    # Top 10 CRYPTO (24/7 validated)
    print("\nTop 10 CRYPTO (24/7 validated) features:")
    crypto_top = stability_df[stability_df['dataset']=='crypto_247'].head(25)
    if len(crypto_top) > 0:
        for idx, row in crypto_top.iterrows():
            print(f"  {row['feature']:45s} stability: {row['stability_pct']:5.0f}% (mkt: {row['market_appearances']}/{n_runs}, 24/7: {row['full_appearances']}/{n_runs}), avg_MI: {row['avg_mi_combined']:.5f}")
    else:
        print("  (None)")
    
    # Threshold analysis
    thresholds_pct = [100, 90, 80, 70, 60, 50, 40, 30, 20, 10]
    n_features_at_threshold = []
    n_market_only_at_threshold = []
    n_crypto_at_threshold = []
    
    for thresh in thresholds_pct:
        n_total = len(stability_df[stability_df['stability_pct'] >= thresh])
        n_market = len(stability_df[(stability_df['stability_pct'] >= thresh) & (stability_df['dataset'] == 'market_only')])
        n_crypto = len(stability_df[(stability_df['stability_pct'] >= thresh) & (stability_df['dataset'] == 'crypto_247')])
        
        n_features_at_threshold.append(n_total)
        n_market_only_at_threshold.append(n_market)
        n_crypto_at_threshold.append(n_crypto)
    
    # Visualization for THIS target
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Total features by threshold
    axes[0].plot(thresholds_pct, n_features_at_threshold, marker='o', linewidth=3, markersize=10, color='#3498db')
    axes[0].set_xlabel('Stability Threshold (%)', fontsize=12)
    axes[0].set_ylabel('Number of Features', fontsize=12)
    axes[0].set_title(f'{target_name}: Total Features', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    axes[0].invert_xaxis()
    
    for x, y in zip(thresholds_pct[::2], n_features_at_threshold[::2]):
        axes[0].text(x, y + max(n_features_at_threshold)*0.02, f'{y}', ha='center', fontsize=9, fontweight='bold')
    
    # Plot 2: Breakdown by dataset
    axes[1].plot(thresholds_pct, n_market_only_at_threshold, marker='s', linewidth=2.5, markersize=8, 
                 label='Market Hours Only', color='#2ecc71')
    axes[1].plot(thresholds_pct, n_crypto_at_threshold, marker='^', linewidth=2.5, markersize=8, 
                 label='Crypto (24/7)', color='#9b59b6')
    axes[1].set_xlabel('Stability Threshold (%)', fontsize=12)
    axes[1].set_ylabel('Number of Features', fontsize=12)
    axes[1].set_title(f'{target_name}: By Context', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    axes[1].invert_xaxis()
    
    plt.tight_layout()
    plt.show()
    
    # Store results
    all_target_results[target_name] = {
        'stability_df': stability_df,
        'market_mi_scores': market_mi_scores,
        'full_mi_scores': full_mi_scores
    }
    
    # Cleanup for this iteration
    del dataset_market, y_market, X_market, X_market_valid, y_market_valid
    del dataset_247, y_247, X_247, X_247_filtered, X_247_valid, y_247_valid
    del market_mi_scores, market_appearances, full_mi_scores, full_appearances
    del stability_df, market_only_features, crypto_247_features
    gc.collect()

print("\n✓ MI selection complete for all targets!")
print(f"✓ Results stored in 'all_target_results' dictionary")

In [ ]:
# ===== CROSS-TARGET FEATURE SELECTION =====
print("\n" + "="*80)
print("CROSS-TARGET FEATURE SELECTION")
print("="*80)

# Aggregate feature stability AND MI scores across all targets
feature_target_matrix = {}

for target_name in selected_targets:
    stability_df = all_target_results[target_name]['stability_df']
    for _, row in stability_df.iterrows():
        feature = row['feature']
        if feature not in feature_target_matrix:
            feature_target_matrix[feature] = {
                'stabilities': [],
                'mi_market': [],
                'mi_247': [],
                'mi_combined': [],
                'market_appearances': [],
                'full_appearances': [],
                'dataset': row['dataset']
            }
        feature_target_matrix[feature]['stabilities'].append(row['stability_pct'])
        feature_target_matrix[feature]['mi_market'].append(row['avg_mi_market'])
        feature_target_matrix[feature]['mi_247'].append(row['avg_mi_247'])
        feature_target_matrix[feature]['mi_combined'].append(row['avg_mi_combined'])
        feature_target_matrix[feature]['market_appearances'].append(row['market_appearances'])
        feature_target_matrix[feature]['full_appearances'].append(row['full_appearances'])

# Create summary DataFrame
cross_target_summary = []
for feature, data in feature_target_matrix.items():
    stabilities = data['stabilities']
    mi_market = data['mi_market']
    mi_247 = data['mi_247']
    mi_combined = data['mi_combined']
    market_apps = data['market_appearances']
    full_apps = data['full_appearances']
    
    # Calculate separate stability for market and 24/7 for crypto features
    if data['dataset'] == 'crypto_247':
        # Market stability: average of (market_appearances / n_runs) across targets
        stability_market = np.mean([m / 10 * 100 for m in market_apps])
        # 24/7 stability: average of (full_appearances / n_runs) across targets
        stability_247 = np.mean([f / 10 * 100 for f in full_apps])
    else:
        stability_market = np.mean(stabilities)
        stability_247 = 0
    
    cross_target_summary.append({
        'feature': feature,
        'n_targets': len(stabilities),
        'avg_stability': np.mean(stabilities),
        'stability_market': stability_market,
        'stability_247': stability_247,
        'avg_mi_market': np.mean(mi_market),
        'avg_mi_247': np.mean(mi_247),
        'avg_mi_combined': np.mean(mi_combined),
        'dataset': data['dataset']
    })

cross_target_df = pd.DataFrame(cross_target_summary)

# Sort by avg_mi_combined (predictive power)
cross_target_df = cross_target_df.sort_values('avg_mi_combined', ascending=False)

print(f"\nTotal unique features: {len(cross_target_df)}")

# Show top features in each category
print("\n" + "="*80)
print("TOP 50 FEATURES (ranked by average MI across all 6 targets)")
print("="*80)

print("\nMARKET-HOURS features:")
market_top = cross_target_df[cross_target_df['dataset']=='market_only'].head(15)
for idx, row in market_top.iterrows():
    print(f"  {row['feature']:50s} MI: {row['avg_mi_market']:.5f}, stability: {row['avg_stability']:5.1f}%")

print("\nCRYPTO (24/7) features:")
crypto_top = cross_target_df[cross_target_df['dataset']=='crypto_247'].head(15)
for idx, row in crypto_top.iterrows():
    print(f"  {row['feature']:50s} │ MKT: MI={row['avg_mi_market']:.5f} stab={row['stability_market']:5.1f}% │ 24/7: MI={row['avg_mi_247']:.5f} stab={row['stability_247']:5.1f}%")

# ===== NEW: MULTIPLE THRESHOLD VIEWS =====
print("\n" + "="*80)
print("FEATURE COUNTS AT DIFFERENT THRESHOLDS")
print("="*80)

# Test different threshold combinations
# Add these additional thresholds to see the distribution better:
thresholds_to_test = [
    (80, 0.010),
    (70, 0.010),
    (60, 0.010),
    (50, 0.010),
    (80, 0.005),
    (70, 0.005),
    (60, 0.005),
    (50, 0.005),
    (70, 0.003),
    (60, 0.003),
    (50, 0.003),
    (60, 0.002),  # ADD THIS
    (50, 0.002),  # ADD THIS
    (60, 0.001),  # ADD THIS
    (50, 0.001),  # ADD THIS
]

print("\nMARKET-HOURS features:")
market_subset = cross_target_df[cross_target_df['dataset']=='market_only']
for stab_thresh, mi_thresh in thresholds_to_test:
    count = len(market_subset[(market_subset['avg_stability'] >= stab_thresh) & 
                              (market_subset['avg_mi_combined'] >= mi_thresh)])
    print(f"  ≥{stab_thresh}% stability AND ≥{mi_thresh:.3f} MI: {count:3d} features")

print("\nCRYPTO (24/7) features:")
crypto_subset = cross_target_df[cross_target_df['dataset']=='crypto_247']
for stab_thresh, mi_thresh in thresholds_to_test:
    count = len(crypto_subset[(crypto_subset['avg_stability'] >= stab_thresh) & 
                              (crypto_subset['avg_mi_combined'] >= mi_thresh)])
    print(f"  ≥{stab_thresh}% stability AND ≥{mi_thresh:.3f} MI: {count:3d} features")


In [ ]:
# ===== FEATURE SELECTION AND DATASET PREPARATION =====
print("\n" + "="*80)
print("FEATURE SELECTION")
print("="*80)

# Select features based on OPTIMIZED thresholds (different for market vs crypto)
market_subset = cross_target_df[cross_target_df['dataset'] == 'market_only']
crypto_subset = cross_target_df[cross_target_df['dataset'] == 'crypto_247']

# Market-hours: 70% stability + 0.005 MI
features_market = market_subset[
    (market_subset['avg_stability'] >= 80) & 
    (market_subset['avg_mi_combined'] >= 0.005)
]['feature'].tolist()

# Crypto 24/7: 60% stability + 0.002 MI
features_crypto = crypto_subset[
    (crypto_subset['avg_stability'] >= 60) & 
    (crypto_subset['avg_mi_combined'] >= 0.004)
]['feature'].tolist()

# Combine all selected features
features_all = features_market + features_crypto

print(f"\nSelection Criteria:")
print(f"  Market-hours: ≥70% stability AND ≥0.005 MI")
print(f"  Crypto 24/7:  ≥60% stability AND ≥0.002 MI")

print(f"\nSelected Features:")
print(f"  Market-hours: {len(features_market)}")
print(f"  Crypto (24/7): {len(features_crypto)}")
print(f"  TOTAL:        {len(features_all)}")

print(f"\nTargets: {len(target_cols)}")

# Add targets to columns
all_cols_to_keep = features_all + target_cols

# Filter datasets
merged_common_selected = merged_common_clean[all_cols_to_keep]
merged_all_selected = merged_all_clean[all_cols_to_keep]

print(f"\nDatasets:")
print(f"  merged_common_selected: {merged_common_selected.shape}")
print(f"  merged_all_selected:    {merged_all_selected.shape}")

# Summary statistics
print("\n" + "="*80)
print("FEATURE QUALITY SUMMARY")
print("="*80)

print(f"\nMarket-hours features:")
print(f"  Avg MI:        {market_subset[market_subset['feature'].isin(features_market)]['avg_mi_combined'].mean():.5f}")
print(f"  Avg stability: {market_subset[market_subset['feature'].isin(features_market)]['avg_stability'].mean():.1f}%")

print(f"\nCrypto 24/7 features:")
print(f"  Avg MI:        {crypto_subset[crypto_subset['feature'].isin(features_crypto)]['avg_mi_combined'].mean():.5f}")
print(f"  Avg stability: {crypto_subset[crypto_subset['feature'].isin(features_crypto)]['avg_stability'].mean():.1f}%")

print("\n✓ Ready for training!")

Get Data

In [ ]:
import pickle

# ===== SAVE DATASETS AND FEATURE LISTS =====
print("\n" + "="*80)
print("SAVING FILES")
print("="*80)

# Save datasets (no timestamp)
with open('merged_common_selected.pkl', 'wb') as f:
    pickle.dump(merged_common_selected, f)

with open('merged_all_selected.pkl', 'wb') as f:
    pickle.dump(merged_all_selected, f)

# Save feature lists (no timestamp)
with open('features_all.pkl', 'wb') as f:
    pickle.dump(features_all, f)

with open('features_market.pkl', 'wb') as f:
    pickle.dump(features_market, f)

with open('features_crypto.pkl', 'wb') as f:
    pickle.dump(features_crypto, f)

print(f"\n✓ Saved 2 datasets + 3 feature lists")

OLD MI CODE

In [ ]:
"""
from sklearn.feature_selection import mutual_info_classif
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===== SET DATASET HERE =====
dataset = dataset_clean  # Uses the cleaned dataset from previous step
# ============================

# Prepare data
target_cols = ['target_n1', 'target_n3', 'target_n5', 'target_sharpe_n3']
y = dataset['target_n3']
X_all = dataset.drop(columns=target_cols).select_dtypes(include=[np.number])
X_all = X_all.replace([np.inf, -np.inf], np.nan)

X_valid = X_all[y.notna()]
y_valid = y[y.notna()]

print(f"Total features: {X_valid.shape[1]}")
print(f"Total rows: {len(X_valid):,}\n")

# Run MI multiple times
n_runs = 10
sample_size = 20000
feature_counts = {feature: 0 for feature in X_valid.columns}

print(f"Running {n_runs} MI runs...")
for i in range(n_runs):
    # Random sample
    sample_idx = np.random.RandomState(42+i).choice(len(X_valid), sample_size, replace=False)
    X_sample = X_valid.iloc[sample_idx].dropna()
    y_sample = y_valid.iloc[sample_idx].loc[X_sample.index]
    
    # Compute MI
    mi_scores = mutual_info_classif(X_sample, y_sample, random_state=42+i)
    
    # Track each feature's rank
    for feature, score in zip(X_sample.columns, mi_scores):
        feature_counts[feature] += 1 if score > 0 else 0
    
    print(f"  Run {i+1}/{n_runs}: {len(X_sample)} clean rows")

# Calculate stability percentage
stability_df = pd.DataFrame({
    'feature': list(feature_counts.keys()),
    'appearances': list(feature_counts.values()),
    'stability_pct': [v/n_runs*100 for v in feature_counts.values()]
}).sort_values('stability_pct', ascending=False)

# Check different thresholds
thresholds_pct = [100, 90, 80, 70, 60, 50, 40, 30, 20, 10]
n_features_at_threshold = []

print("\n" + "="*80)
for thresh in thresholds_pct:
    n = len(stability_df[stability_df['stability_pct'] >= thresh])
    n_features_at_threshold.append(n)
    print(f"≥{thresh:3d}% stability: {n:4d} features")

# Plot
plt.figure(figsize=(10, 6))
plt.plot(thresholds_pct, n_features_at_threshold, marker='o', linewidth=3, markersize=10)
plt.xlabel('Stability Threshold (%)', fontsize=13)
plt.ylabel('Number of Features', fontsize=13)
plt.title('How Many Features to Keep Based on MI Stability', fontsize=15, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.gca().invert_xaxis()

# Add labels
for x, y in zip(thresholds_pct, n_features_at_threshold):
    plt.text(x, y + 15, f'{y}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print(f"RECOMMENDATION: Pick a threshold where you get 150-300 features")
print(f"These are features that consistently show up across different data samples")
"""